# F5 — prompted baseline arms (`base_fewshot`, `base_fewshot_constrained`)

Runs un-finetuned `Qwen/Qwen3-1.7B` over the 300 `eval_gold` documents twice and writes two
prediction files for `sxl metrics score` to consume on the laptop.

**Before running:**

1. Settings → Accelerator → **`GPU T4 x2`** (the code uses `cuda:0` only, so latency stays
   comparable across arms — SPEC §2.2).
2. Settings → Internet **on** (the pip install and the model download need it).
3. Add Data → the **`sxl-data`** dataset (`eval_gold.jsonl`, `train.jsonl`, `dev.jsonl`).
4. Set `COMMIT_SHA` in the install cell to the commit you want to measure.

This notebook is thin on purpose: every line of logic lives in `src/sxl/` and is unit-tested
on the laptop. **Stop the session manually when it finishes — idle sessions burn quota.**

In [ ]:
# HF_HOME must be set BEFORE anything imports huggingface_hub: the cache path is
# frozen into module constants at import time. The default lives on /kaggle/working,
# which is only 20 GB and is the persisted notebook output — a 3.4 GB model plus
# safetensors staging would fill it. /kaggle/tmp is ~60 GB of scratch (SPEC §2.2).
import os

os.environ["HF_HOME"] = "/kaggle/tmp/hf"
os.makedirs("/kaggle/tmp/hf", exist_ok=True)

import time

SESSION_STARTED = time.time()  # GPU-hour accounting; printed in the last cell

In [ ]:
# Pin a commit SHA, never a branch (SPEC §2.4): a mid-session push must not change
# what a running notebook is executing.
COMMIT_SHA = "0000000000000000000000000000000000000000"  # <- paste `git rev-parse HEAD`

# Shape check, not a placeholder comparison: a find-and-replace over this cell
# would rewrite both copies of a placeholder string and silently disarm the guard.
import re

assert re.fullmatch(r"[0-9a-f]{40}", COMMIT_SHA)
assert set(COMMIT_SHA) != {"0"}

REPO = "https://github.com/RazaAli1010/schema-extract-lab"

# `schema-extract-lab`, not `sxl` — that is the DISTRIBUTION name from pyproject.toml.
# `sxl` is only the import package and the console script, and pip resolves the
# direct reference by distribution name (it will go looking on PyPI otherwise).
#
# No flash-attn (needs sm80) and no vLLM (banned, SPEC §5.3 — Turing support is degrading).
!pip install -q "schema-extract-lab[gpu] @ git+{REPO}@{COMMIT_SHA}"

# Kaggle preinstalls torchvision/torchaudio built against ITS torch (2.10.0). Our
# pinned torch==2.13.0 leaves their compiled ops unregistered, so importing
# torchvision raises `operator torchvision::nms does not exist` — and transformers
# imports it eagerly via `is_torchvision_available()`, which turns into a baffling
# `Could not import module 'Qwen3ForCausalLM'`. Neither package is used here.
#
# RESTART THE SESSION after this cell the first time: a failed torchvision import
# leaves broken entries in sys.modules that uninstalling does not clear. Installed
# packages survive the restart, so the cell above becomes a fast no-op.
!pip uninstall -q -y torchvision torchaudio

In [ ]:
# Version banner (SPEC §5.7). Printed into the saved output so a stale Kaggle base
# image is visible in the artifact rather than being a mystery six weeks later.
import importlib.metadata as md

import torch

# "schema-extract-lab" is the distribution name; "sxl" is only the import package.
for pkg in ("torch", "transformers", "trl", "peft", "bitsandbytes", "outlines", "schema-extract-lab"):
    try:
        print(f"{pkg:>20} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:>20} NOT INSTALLED")

name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
print(f"\n{name}  capability={capability}  n_gpus={torch.cuda.device_count()}")

# Fail loudly rather than quietly measuring a P100: every latency and cost number
# downstream is labelled "Tesla T4", and sm_75 is also what forces fp16 + sdpa.
assert capability == (7, 5), f"expected a T4 (7, 5), got {capability} on {name}"

!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# The package is a pip install here, so `config.ROOT` points into site-packages and
# the default data paths are wrong. Pass all of them explicitly (see config.py).
#
# The mount point is discovered rather than hard-coded: depending on how a dataset
# is attached, Kaggle mounts it at /kaggle/input/<slug>/ OR at
# /kaggle/input/datasets/<owner>/<slug>/, and guessing wrong costs a session.
import glob

matches = glob.glob("/kaggle/input/**/eval_gold.jsonl", recursive=True)
assert matches, "sxl-data is not attached — use Add Input in the sidebar"
assert len(matches) == 1, f"eval_gold.jsonl found in several places: {matches}"

DATA = os.path.dirname(matches[0])
GOLD = f"{DATA}/eval_gold.jsonl"
TRAIN = f"{DATA}/train.jsonl"
OUT = "/kaggle/working/predictions"

assert os.path.exists(TRAIN), f"train.jsonl missing from {DATA} — the few-shot exemplars live there"
os.makedirs(OUT, exist_ok=True)

print(f"DATA = {DATA}")
!wc -l {GOLD} {TRAIN}

In [ ]:
# Smoke run first: 8 documents, ~1 minute. Inspect the raw output by eye before
# spending 15 minutes on the full arm.
!sxl gpu predict --arm base_fewshot --limit 8 --gold {GOLD} --train {TRAIN} --out /kaggle/working/_smoke.jsonl

In [ ]:
# What did it actually say? A schema-shaped object with no prose and no <think>
# block is what a healthy run looks like.
import json

with open("/kaggle/working/_smoke.jsonl", encoding="utf-8") as fh:
    smoke = [json.loads(line) for line in fh]

print(f"{sum(r['schema_valid'] for r in smoke)}/{len(smoke)} schema-valid\n")
print(smoke[0]["raw_output"][:1500])

In [ ]:
# Arm 1 of 2: greedy, unconstrained. The competitor that matters (SPEC §3.6).
# Resumable — if the session dies, re-run this cell and it picks up from the
# .partial.jsonl file without regenerating completed documents.
!sxl gpu predict --arm base_fewshot --gold {GOLD} --train {TRAIN} --out {OUT}/base_fewshot.jsonl

In [ ]:
# Arm 2 of 2: Outlines schema-constrained decoding. Slower per token (logit masking)
# and sequential, so budget appreciably longer than arm 1. Expect schema_valid_rate
# ~1.0 and macro_f1 close to arm 1 — that gap is the point of the arm.
!sxl gpu predict --arm base_fewshot_constrained --gold {GOLD} --train {TRAIN} --out {OUT}/base_fewshot_constrained.jsonl

In [ ]:
# Acceptance criteria, asserted in the artifact itself.
ARMS = ("base_fewshot", "base_fewshot_constrained")
gold_ids = [json.loads(line)["doc_id"] for line in open(GOLD, encoding="utf-8")]

for arm in ARMS:
    with open(f"{OUT}/{arm}.jsonl", encoding="utf-8") as fh:
        records = [json.loads(line) for line in fh]

    # Hard contract — these are what F4 depends on, so they stay assertions.
    assert len(records) == 300, (arm, len(records))
    assert [r["doc_id"] for r in records] == gold_ids, f"{arm}: doc_ids drifted from eval_gold"
    assert not any("<think>" in r["raw_output"] for r in records), f"{arm}: thinking mode leaked"

    n_valid = sum(r["schema_valid"] for r in records)
    n_trunc = sum(r["completion_tokens"] >= 512 for r in records)
    print(f"{arm:>26}  valid {n_valid}/300 ({n_valid / 3:.1f}%)  truncated {n_trunc}")

# The constrained arm was EXPECTED to reach >= 0.99, on the assumption that a
# grammar makes invalid output impossible. The 8-doc smoke showed it does not: the
# grammar permits an arbitrarily long `required_skills` list, so a model that loops
# runs past MAX_NEW_TOKENS and the truncated object fails to parse. That is a real
# cost of constrained decoding (F5 §Implementation notes predicted it), so this is
# reported rather than asserted — halting here would discard a valid measurement.
# F8 must state the measured rate and this cause.
constrained = [json.loads(line) for line in open(f"{OUT}/base_fewshot_constrained.jsonl")]
rate = sum(r["schema_valid"] for r in constrained) / len(constrained)
print(f"\nbase_fewshot_constrained schema_valid_rate = {rate:.3f}  (expected >= 0.99)")
if rate < 0.99:
    trunc = [r for r in constrained if not r["schema_valid"] and r["completion_tokens"] >= 512]
    print(f"  MISSED. {len(trunc)} of the invalid rows are truncation at max_new_tokens.")

print(f"\nelapsed: {(time.time() - SESSION_STARTED) / 3600:.2f} GPU-hours (budget: < 4)")
!ls -la {OUT}

## Back on the laptop

Download `predictions/base_fewshot.jsonl` and `predictions/base_fewshot_constrained.jsonl`
from this notebook's output into `artifacts/predictions/`, then:

```bash
sxl metrics score --arm base_fewshot
sxl metrics score --arm base_fewshot_constrained
sxl metrics compare
```

Whatever the numbers are, they ship (SPEC §1.1). Do not tune the prompt to move them.

**Now stop the session** (Run → Stop session) — idle sessions keep burning the weekly quota.